# Подготовка, обучение и разбор событий в одном ноутбуке

Выполняйте ячейки сверху вниз. Настройки, аудит режимов, реестр событий, обучение, тревоги и интерпретация выводятся здесь. Отдельно запускать `.py` и редактировать отчёты в CSV не требуется.

Реальные данные остаются у вас. Причины событий и назначения датчиков требуют проверки специалистом; отсутствие достаточной разметки выводится как отдельный статус.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
from IPython.display import display

_candidates = [Path.cwd(), Path.cwd()/"training"]
if "__file__" in globals():
    _candidates.insert(0, Path(__file__).resolve().parent)
TRAINING_DIR = next((p for p in _candidates if (p/"lib/monitoring_v3").is_dir()), None)
if TRAINING_DIR is None:
    raise FileNotFoundError("Откройте ноутбук из папки проекта или training.")
os.chdir(TRAINING_DIR)
for _path in (TRAINING_DIR, TRAINING_DIR/"lib"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

from monitoring_v3.config import load
from monitoring_v3.data import load_sources
import monitoring_v3.pipeline as LF
from notebook_reports import (
    configure, show_table, show_preparation, show_results, show_detail,
    episode_catalog, apply_episode_labels, run_interpretation, run_research,
)


## 1. Все настройки

При обычном открытии берутся начальные настройки из `incremental_settings.json`. Любое значение можно переопределить в `SETTINGS` ниже. Правила качества также задаются здесь — исходный файл правил сохранится.

Изменение порога, правил или состава признаков требует новой папки `output`, чтобы не продолжать несовместимую цепочку обучения.

In [ ]:
SETTINGS = {
    # "data_paths": {"К-5": "D:/private/К-5/hourly.parquet"},
    # "groups": {"БКЗ-500": ["К-5", "К-6", "К-7"]},
    # "first_test": "2025-03-01",
    # "run_thresholds": {"К-5": ...},  # физически проверенный порог
    # "startup_hours": 24,
    # "output": "runs/my_experiment",
}
SENSOR_RULES = {
    "defaults": {},
    "sensors": {
        # "общий параметр из аудита": {"kind": "rare", "mask_frozen": False},
    },
}
DETAIL = {
    "experiment": None,  # None — первый рассчитанный эксперимент; названия будут в таблице
    "unit": None,        # None — первый доступный котёл
    "view": "mean::instant",
    "budget": None,      # None — основной бюджет из настроек
    "start": None,       # например "2025-03-01"
    "end": None,
}
INTERPRETATION = {"run": True, "as_of": None, "attach": False}
# as_of=None — последняя доступная дата телеметрии данного котла, не сегодняшняя дата.
# attach=True — добавить обученные классификаторы в экспорт выбранных экспериментов.
RESEARCH = {
    "ensemble": False, "survival": False, "sensitivity": False,
    "as_of": None, "sensor": None, "start": None,
}

CFG = load(os.environ.get("BOILER_CONFIG_JSON", "incremental_settings.json"))
CFG.update(SETTINGS)
CFG = configure(CFG, SENSOR_RULES)
show_table("Действующие настройки", pd.DataFrame(
    {"параметр": list(CFG), "значение": [str(value) for value in CFG.values()]}))


## 2. Данные, режимы и события до обучения

Ниже появятся датчик нагрузки и порог каждого котла, часы работы/пуска/останова/неизвестного состояния, доступность датчиков, полный реестр ремонтов и резервов, графики нагрузки и режимов.

Большие таблицы можно раскрыть нажатием на название. Ошибки источников нужно исправить в `SETTINGS` и повторить эту ячейку.

In [ ]:
CONTEXTS, ISSUES = load_sources(CFG, Path(CFG["output"])/"audit", strict=False)
EVENTS = show_preparation(CONTEXTS, ISSUES, CFG)


## 3. Последовательное обучение и обновление

Первое обучение и последующие месяцы выполняются автоматически. Готовые совместимые checkpoints восстанавливаются. Если для нового месяца недостаточно подходящих данных, используется предыдущая модель с указанием причины.

In [ ]:
if ISSUES:
    raise ValueError("Исправьте проблемы источников в SETTINGS и повторите аудит выше.")
RESULT = LF.run(CFG)


## 4. Результаты моделей и месячных обновлений

Показаны качество схем тревог, сравнение на общих часах, журналы обновлений, качество прогноза и эпизоды. Все строки доступны в раскрывающихся таблицах. Файлы результатов дополнительно сохраняются автоматически.

In [ ]:
EXPERIMENTS = show_results(CFG, RESULT)
show_table("Эксперименты для выбора в DETAIL", EXPERIMENTS.drop(columns="directory", errors="ignore"), True)


## 5. График и события выбранного эксперимента

Выбор котла, модели, периода и схемы — в `DETAIL` в разделе настроек. График обрезается без пересчёта тревог. Ниже — обнаружение каждой аварии и эпизоды тревог с фактическим временем подтверждения.

In [ ]:
DETAIL_DIR = show_detail(CFG, DETAIL)


## 6. Проверка смысла эпизодов прямо в ноутбуке

Скопируйте идентификатор из каталога и добавьте запись в `EPISODE_LABEL_EDITS` ниже. Укажите эксперимент, котёл, смысл, специалиста и дату, когда эта метка стала известна. Это сохраняет вашу разметку автоматически; открывать CSV не требуется.

`suggestions` — гипотезы для проверки, а не подтверждённые причины. После изменения списка выполните ячейку сохранения меток и следующую ячейку интерпретации.

In [ ]:
CATALOG = episode_catalog(CFG)
show_table("Каталог эпизодов для разбора", CATALOG, True)


In [ ]:
EPISODE_LABEL_EDITS = [
    # {
    #     "experiment": "название из таблицы", "unit": "К-5",
    #     "episode_id": "идентификатор из каталога", "label": "mode_change",
    #     "label_available_at": "2025-04-01", "reviewer": "специалист",
    #     "reason": "подтверждено переключение оборудования",
    # },
]
if EPISODE_LABEL_EDITS:
    CATALOG = apply_episode_labels(CFG, EPISODE_LABEL_EDITS)
    show_table("Сохранённые метки", CATALOG.loc[CATALOG.label.notna()], True)
else:
    print("Новых проверенных меток нет; прежняя разметка сохранена.")


## 7. Интерпретация отклонений и предаварийного риска

Выполняется в этой ячейке для рассчитанных экспериментов. Если независимых событий или проверенных меток мало, таблица объяснит причину, а смысл останется неопределённым. Отключить новое обучение и показать сохранённые статусы можно через `INTERPRETATION["run"] = False`.

In [ ]:
INTERPRETATION_RESULT = run_interpretation(
    CFG, CONTEXTS, enabled=INTERPRETATION["run"],
    as_of=INTERPRETATION["as_of"], attach=INTERPRETATION["attach"],
)


## 8. Дополнительные исследования без отдельных команд

В `RESEARCH` можно включить сбор признаков нескольких детекторов (`ensemble`), исследование времени до отказа (`survival`) и проверку искусственными нарушениями (`sensitivity`). Для последней укажите `sensor` и `start`; используется модель из `DETAIL` и копия истории. Эти проверки не подтверждают качество на реальных авариях.

Вывод и таблицы появятся непосредственно под ячейкой.

In [ ]:
RESEARCH_RESULT = run_research(CFG, CONTEXTS, RESEARCH, DETAIL_DIR)
